# Study 880 — Aggregate Short Interest — the teardown

The horizon sweep, the Newey-West slope *t*, the high/low tercile split, the 5,000-draw permutation placebo, the two-era cut, the costed timing overlay, and the 20-seed synthetic control. FINRA consolidated short interest, 50-name liquid panel, equal-weight mean days-to-cover; SPY total-return.

In [1]:
R = {'start': '2017-12-29', 'end': '2026-06-30', 'n_dates': 205, 'n_panel': 50, 'fingerprint': '4ab933bfbc01', 'last_index': 2.146, 'median_names': 50, 'h1_n': 203, 'h1_beta': -17.2, 'h1_t': -0.66, 'h1_r2': 0.28, 'h1_fwd': 55, 'h2_n': 202, 'h2_beta': -12.8, 'h2_t': -0.26, 'h2_r2': 0.08, 'h2_fwd': 110, 'h3_n': 201, 'h3_beta': -35.5, 'h3_t': -0.5, 'h3_r2': 0.41, 'h3_fwd': 168, 'h6_n': 198, 'h6_beta': -101.3, 'h6_t': -0.7, 'h6_r2': 2.07, 'h6_fwd': 338, 'terc_lo': 67, 'terc_hi': 21, 'terc_welch': -0.75, 'terc_n': 68, 'placebo_obs': -17.22, 'placebo_mean': -0.13, 'placebo_sd': 22.48, 'placebo_p': 0.217, 'era_e_n': 107, 'era_e_beta': -21.4, 'era_e_t': -0.6, 'era_e_r2': 0.5, 'era_l_n': 96, 'era_l_beta': -13.2, 'era_l_t': -0.34, 'era_l_r2': 0.13, 'timer1_net': 35.3, 'timer1_ann': 8.5, 'timer1_sharpe': 0.71, 'timer1_sw': 70, 'timer5_net': 34.0, 'timer5_ann': 8.2, 'timer5_sharpe': 0.68, 'bh_ann': 13.2, 'null_mean_t': 0.47, 'null_sd_t': 0.72, 'null_fire': 1, 'null_100_mean': 0.02, 'planted_beta': -103.8, 'planted_t': -4.1, 'planted_r2': 7.0}

## The headline — forward-SPY-return regression on the detrended index

One publication lag (signal at settlement `t` acted on the next settlement `t+1`). Horizons in bi-monthly settlement periods (~0.5 mo each). RRZ predict beta < 0.

In [2]:
for h in ('h1','h2','h3','h6'):
    print(f"H={h[1:]:>1} (~{int(h[1:])*0.5:.1f} mo): n={R[h+'_n']:3d}  "
          f"beta={R[h+'_beta']:+7.1f} bps/sigma  NW t={R[h+'_t']:+.2f}  "
          f"R2={R[h+'_r2']:+.2f}%  fwd mean={R[h+'_fwd']:+d} bps")

H=1 (~0.5 mo): n=203  beta=  -17.2 bps/sigma  NW t=-0.66  R2=+0.28%  fwd mean=+55 bps
H=2 (~1.0 mo): n=202  beta=  -12.8 bps/sigma  NW t=-0.26  R2=+0.08%  fwd mean=+110 bps
H=3 (~1.5 mo): n=201  beta=  -35.5 bps/sigma  NW t=-0.50  R2=+0.41%  fwd mean=+168 bps
H=6 (~3.0 mo): n=198  beta= -101.3 bps/sigma  NW t=-0.70  R2=+2.07%  fwd mean=+338 bps


## High vs low short-interest tercile — forward one-period SPY return

In [3]:
print(f"low-SI tercile  {R['terc_lo']:+d} bps  (n={R['terc_n']})")
print(f"high-SI tercile {R['terc_hi']:+d} bps  (n={R['terc_n']})  "
      f"Welch t(high-low) = {R['terc_welch']:+.2f}")

low-SI tercile  +67 bps  (n=68)
high-SI tercile +21 bps  (n=68)  Welch t(high-low) = -0.75


## Placebo — permute forward returns vs the index (5,000 draws)

In [4]:
print(f"observed beta {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.2f} "
      f"(sd {R['placebo_sd']:.2f}) -> left-tail p = {R['placebo_p']:.4f}")

observed beta -17.22 bps vs placebo mean -0.13 (sd 22.48) -> left-tail p = 0.2170


## Robustness — two eras (split 2022-06-01)

In [5]:
print(f"2017-12 -> 2022-05 (n={R['era_e_n']}): beta={R['era_e_beta']:+.1f} bps  NW t={R['era_e_t']:+.2f}  R2={R['era_e_r2']:.2f}%")
print(f"2022-06 -> 2026-06 (n={R['era_l_n']}): beta={R['era_l_beta']:+.1f} bps  NW t={R['era_l_t']:+.2f}  R2={R['era_l_r2']:.2f}%")

2017-12 -> 2022-05 (n=107): beta=-21.4 bps  NW t=-0.60  R2=0.50%
2022-06 -> 2026-06 (n=96): beta=-13.2 bps  NW t=-0.34  R2=0.13%


## The timer — de-risk to cash when shorts are crowded (`sii>0`)

One-way cost × NAV per switch. The overlay *underperforms* buy-and-hold — sitting out on crowded-short readings just gave up equity premium.

In [6]:
for tag,net,ann,sh in [('1 bp',R['timer1_net'],R['timer1_ann'],R['timer1_sharpe']),
                       ('5 bps',R['timer5_net'],R['timer5_ann'],R['timer5_sharpe'])]:
    print(f"{tag:>5}: overlay net {net:+.1f} bps/period ({ann:+.1f}%/yr, Sharpe {sh:.2f})")
print(f"buy-and-hold: {R['bh_ann']:+.1f}%/yr  (the overlay loses to it)")

 1 bp: overlay net +35.3 bps/period (+8.5%/yr, Sharpe 0.71)
5 bps: overlay net +34.0 bps/period (+8.2%/yr, Sharpe 0.68)
buy-and-hold: +13.2%/yr  (the overlay loses to it)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted relation.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from agg_short import data, strategy as st
nt = np.array([st.synthetic_detect(data.synthetic_frame(edge=0.0, seed=880+s, n_periods=200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: slope NW t mean {nt.mean():+.2f} (sd {nt.std(ddof=1):.2f}), |t|>=2 in {(abs(nt)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_frame(edge=0.015, seed=880, n_periods=200))
print(f"planted (edge=0.015): beta = {planted['beta']*1e4:+.1f} bps/sigma, NW t = {planted['t_nw']:+.2f}, R2 = {planted['r2']*100:.1f}%")

null (edge=0), 8 seeds: slope NW t mean +0.83 (sd 0.77), |t|>=2 in 1/8
planted (edge=0.015): beta = -103.8 bps/sigma, NW t = -4.10, R2 = 7.0%


## Verdict

- **Signal — None.** The RRZ aggregate-short-interest predictor does **not** replicate on a 2017–2026 FINRA-built, mega-cap, days-to-cover index. The forward-return slope has the **right sign** (-17.2 bps/σ, negative at all four horizons) but NW *t* = **-0.66** (R² 0.28%); both eras agree in sign yet neither is significant (*t* = -0.60 / -0.34); the permutation placebo puts it at *p* = 0.22. The 20-seed synthetic control fires on the planted world (*t* = -4.10) and stays quiet on the null, so the flat real result is genuine.
- **Tradability — Mirage.** The de-risk-on-crowded-shorts overlay earns +8.5%/yr net (1 bp) — *below* buy-and-hold's +13.2%/yr; a weak, wrong-way-for-a-bull-market timing rule is a paycheck mirage.

*Caveats travelling with every number: aggregate SI is bi-monthly with an ~8-day publication lag; the index is a days-to-cover average not the paper's shares-outstanding ratio; the panel is current-membership mega-caps (survivorship, named on the Signal axis); and the sample is short (~8.5 years, a mostly-bull era) versus RRZ's 1973–2014.*